In [1]:
import pandas as pd
df = pd.read_csv("CRMLSListing_Week6_Features.csv")
print(df.shape)

(146049, 71)


In [2]:
flag_cols = [
    "listing_after_close_flag",
    "purchase_after_close_flag",
    "negative_timeline_flag",
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_bedrooms_flag",
    "invalid_bathrooms_flag",
    "missing_geo_flag",
    "zero_geo_flag",
    "invalid_longitude_flag",
    "out_of_ca_flag"
]

df[flag_cols].sum().sort_values(ascending=False)

missing_geo_flag               15671
listing_after_close_flag           0
purchase_after_close_flag          0
negative_timeline_flag             0
invalid_close_price_flag           0
invalid_living_area_flag           0
invalid_days_on_market_flag        0
invalid_bedrooms_flag              0
invalid_bathrooms_flag             0
zero_geo_flag                      0
invalid_longitude_flag             0
out_of_ca_flag                     0
dtype: int64

In [3]:
# Convert numeric columns
numeric_cols = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

print("\nNumeric conversion complete")
print(df[numeric_cols].dtypes)


Numeric conversion complete
ClosePrice      float64
LivingArea      float64
DaysOnMarket      int64
dtype: object


In [4]:
# Read existing rule flags
# These flags were already created in previous weeks

rule_flags = [
    "invalid_close_price_flag",
    "invalid_living_area_flag",
    "invalid_days_on_market_flag",
    "invalid_numeric_flag"
]

for col in rule_flags:

    if col in df.columns:

        # Convert to Boolean
        df[col] = (
            df[col]
            .astype(str)
            .str.lower()
            .isin(["true", "1", "yes"])
        )
        print(f"{col} loaded successfully")

    else:
        print(f"Warning: {col} not found")


invalid_close_price_flag loaded successfully
invalid_living_area_flag loaded successfully
invalid_days_on_market_flag loaded successfully


In [5]:
#  Define IQR outlier function

def add_iqr_flag(df, column_name):

    # Calculate quartiles
    Q1 = df[column_name].quantile(0.25)
    Q3 = df[column_name].quantile(0.75)

   
    IQR = Q3 - Q1

  
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

   
    flag_col = f"{column_name}_iqr_outlier_flag"
 # Flag outliers
    df[flag_col] = (
        (df[column_name] < lower) |
        (df[column_name] > upper)
    )

    # Print IQR summary
    print("\n===================================")
    print(f"{column_name} IQR Summary")
    print("===================================")

    print(f"Q1: {Q1}")
    print(f"Q3: {Q3}")
    print(f"IQR: {IQR}")

    print(f"Lower Bound: {lower}")
    print(f"Upper Bound: {upper}")

    print(
        f"Outliers Flagged: {df[flag_col].sum()}"
    )

    return df

In [6]:
# Apply IQR outlier detection
# Apply IQR filtering to:
# ClosePrice
# LivingArea
# DaysOnMarket

df = add_iqr_flag(df, "ClosePrice")

df = add_iqr_flag(df, "LivingArea")

df = add_iqr_flag(df, "DaysOnMarket")


ClosePrice IQR Summary
Q1: 600000.0
Q3: 1350000.0
IQR: 750000.0
Lower Bound: -525000.0
Upper Bound: 2475000.0
Outliers Flagged: 10486

LivingArea IQR Summary
Q1: 1242.0
Q3: 2176.0
IQR: 934.0
Lower Bound: -159.0
Upper Bound: 3577.0
Outliers Flagged: 6308

DaysOnMarket IQR Summary
Q1: 6.0
Q3: 30.0
IQR: 24.0
Lower Bound: -30.0
Upper Bound: 66.0
Outliers Flagged: 12035


In [7]:
# Create combined IQR outlier flag
# If any numeric field is an outlier,
# mark the row as True

df["any_iqr_outlier_flag"] = (

    df["ClosePrice_iqr_outlier_flag"] |

    df["LivingArea_iqr_outlier_flag"] |

    df["DaysOnMarket_iqr_outlier_flag"]

)

print("\nCombined IQR flag created")

print(
    df["any_iqr_outlier_flag"]
    .value_counts()
)



Combined IQR flag created
any_iqr_outlier_flag
False    122232
True      23817
Name: count, dtype: int64


In [8]:
# Create final exclusion flag- Combine:
# rule invalid records + IQR outliers
if "invalid_numeric_flag" in df.columns:

    df["exclude_from_analysis_flag"] = (

        df["invalid_numeric_flag"] |

        df["any_iqr_outlier_flag"]

    )

else:

    df["exclude_from_analysis_flag"] = (
        df["any_iqr_outlier_flag"]
    )

print("\nFinal exclusion flag created")

print(
    df["exclude_from_analysis_flag"]
    .value_counts()
)


Final exclusion flag created
exclude_from_analysis_flag
False    122232
True      23817
Name: count, dtype: int64


In [9]:
#  Create clean filtered dataset
# Keep only records
# that are NOT excluded

df_filtered = df[
    ~df["exclude_from_analysis_flag"]
].copy()

print("\nFiltered dataset created")

print("Original rows:", len(df))

print("Filtered rows:", len(df_filtered))


Filtered dataset created
Original rows: 146049
Filtered rows: 122232


In [10]:
# Compare before and after filtering

comparison_rows = []

fields = [
    "ClosePrice",
    "LivingArea",
    "DaysOnMarket"
]

for col in fields:

    outlier_flag_col = f"{col}_iqr_outlier_flag"

    comparison_rows.append({

        "Field": col,

        "Median Before": df[col].median(),

        "Median After": df_filtered[col].median(),

        "Mean Before": df[col].mean(),

        "Mean After": df_filtered[col].mean(),


    })

comparison = pd.DataFrame(comparison_rows)
print("Before vs After Filtering Comparison")

print(comparison)

Before vs After Filtering Comparison
          Field  Median Before  Median After   Mean Before     Mean After
0    ClosePrice       855000.0      820000.0  1.198568e+06  938345.467573
1    LivingArea         1623.0        1554.0  1.829024e+03    1655.822736
2  DaysOnMarket           12.0          10.0  2.295876e+01      16.353729


In [11]:
# Save outputs
# Save full flagged dataset

df.to_csv(
    "CRMLSListing_Week7_Flagged.csv",
    index=False
)

# Save clean filtered dataset

df_filtered.to_csv(
    "CRMLSListing_Week7_Filtered.csv",
    index=False
)

# Save comparison table

comparison.to_csv(
    "Week7_Listing_Filtering_Comparison.csv",
    index=False
)

In [12]:
print("Saved files:")

print("1. CRMLSListing_Week7_Flagged.csv")

print("2. CRMLSListing_Week7_Filtered.csv")

print("3. Week7_Listing_Filtering_Comparison.csv")

Saved files:
1. CRMLSListing_Week7_Flagged.csv
2. CRMLSListing_Week7_Filtered.csv
3. Week7_Listing_Filtering_Comparison.csv
